# Hallucination Probe - Live Demo

Loads the probe trained by `02_improvements.ipynb` from Google Drive and starts a Gradio app
with a public link. **No retraining**: about 2 minutes from start to link.

**Runtime -> Change runtime type -> T4 GPU**, then **Runtime -> Run all**.


In [ ]:
!pip install -q transformers accelerate bitsandbytes gradio scikit-learn joblib

import joblib, numpy as np, torch
from pathlib import Path
from google.colab import drive
drive.mount("/content/drive")

MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"
WORK = Path("/content/drive/MyDrive/hallucination_probe_v2") / MODEL_ID.split("/")[-1]
saved = joblib.load(WORK / "probe.joblib")
PROBE, POOL, LAYER = saved["probe"], saved["pooling"], saved["layer"]
print(f"probe: pooling={POOL} layer={LAYER} test AUROC={saved.get('auroc', float('nan')):.3f}")


In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

tok = AutoTokenizer.from_pretrained(MODEL_ID)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, device_map="auto",
    quantization_config=BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                                           bnb_4bit_compute_dtype=torch.float16))
model.eval()

# must be IDENTICAL to the prompt used in training (notebook 02)
def build_prompt(question):
    msgs = [{"role": "user",
             "content": "Answer with just the answer, no explanation.\nQ: " + question + "\nA:"}]
    return tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)

@torch.no_grad()
def feature(prompt, answer):
    p_ids = tok(prompt, add_special_tokens=False).input_ids
    a_ids = tok(answer, add_special_tokens=False).input_ids
    if not a_ids:
        return None
    ids = torch.tensor([p_ids + a_ids], device=model.device)
    h = model(input_ids=ids, output_hidden_states=True).hidden_states[LAYER][0]   # (tokens, hidden)
    n = len(p_ids)
    v = {"a_mean": h[n:].mean(0), "a_last": h[-1], "p_last": h[n - 1]}[POOL]
    return v.float().cpu().numpy().astype(np.float16).astype(np.float32).reshape(1, -1)


In [ ]:
import gradio as gr

def ask(question, threshold):
    if not question.strip():
        return "", {}, ""
    p = build_prompt(question)
    enc = tok(p, return_tensors="pt", add_special_tokens=False).to(model.device)
    with torch.no_grad():
        gen = model.generate(**enc, max_new_tokens=20, do_sample=False, pad_token_id=tok.pad_token_id)
    ans = tok.decode(gen[0][enc.input_ids.shape[1]:], skip_special_tokens=True).strip()
    x = feature(p, ans)
    if x is None:
        return ans, {}, "model gave no answer"
    r = float(PROBE.predict_proba(x)[0, 1])
    verdict = (f"NOT CONFIDENT - abstain: \"I'm not sure.\" (risk {r:.2f} > {threshold:.2f})" if r > threshold
               else f"Confident - answer shown (risk {r:.2f} <= {threshold:.2f})")
    return ans, {"likely correct": 1 - r, "likely hallucinated": r}, verdict

gr.Interface(
    fn=ask,
    inputs=[gr.Textbox(label="Question"),
            gr.Slider(0, 1, value=0.6, step=0.05, label="Risk threshold (abstain above this)")],
    outputs=[gr.Textbox(label="Model answer"), gr.Label(label="Probe reading of internal state"),
             gr.Textbox(label="Decision")],
    title="Hallucination Probe - does the model know when it is wrong?",
    description=f"{MODEL_ID}, linear probe on layer {LAYER} ({POOL}). One forward pass, no resampling.",
    examples=[["What is the capital of France?", 0.6],
              ["Who wrote Pride and Prejudice?", 0.6],
              ["Who designed the red telephone box?", 0.6],
              ["Which battle took place on 2 December 1805?", 0.6],
              ["What is the population of Zaqatala district?", 0.6]],
).launch(share=True, debug=True)
